# Projeto Prático: Machine Learning & Inteligência de Mercado
## Análise Estratégica da Concentração no Comércio Global de Bens Criativos (Dataset OpenFCS)

---

> **Componente Curricular:** Machine Learning aplicado à Administração

> **Instituição:** Curso de Graduação em Administração

> **Objetivo:** Aplicação prática de Ciência de Dados, Machine Learning e Inteligência Artificial Generativa para diagnosticar padrões de concentração e (re)configuração competitiva no comércio mundial de bens criativos, a partir do acervo aberto OpenFCS (UNCTAD, alinhado ao UNESCO Framework for Cultural Statistics 2025).

---

### Corpo Docente & Contato

| Atributo | Detalhes |
| :--- | :--- |
| **Professor** | **Sérgio Assunção Monteiro, D.Sc.** |
| **Conecte-se no LinkedIn** | [🌐 linkedin.com/in/sergio-assunção-monteiro](https://www.linkedin.com/in/sergio-assun%C3%A7%C3%A3o-monteiro-b781897b/) |
| **Currículo Lattes** | [🔬 lattes.cnpq.br/9489191035734025](http://lattes.cnpq.br/9489191035734025) |
| **Repositório GitHub** | [💻 github.com/sergiomonteiro76](https://github.com/sergiomonteiro76) |

---

### Sobre este Notebook
Este ambiente foi configurado para que os alunos atuem como **Analistas de Inteligência de Mercado**. Ao longo do semestre, com apoio de modelos de linguagem (IA) integrados ao ecossistema do Google Colab, vamos reconstruir — do dado bruto ao modelo preditivo — o diagnóstico de estrutura competitiva de um setor econômico real: o comércio internacional de bens criativos.

* **Fonte de dados:** [OpenFCS Dataset](https://doi.org/10.5281/zenodo.21211053) — Monteiro & Dubeux (2026), CC-BY-4.0.
* **Material de apoio:** Capítulos 1 a 3 das notas de aula (Ambiente e primeiro contato; Python/pandas/NumPy; Bases de Dados e SQL).

## Aula 10 — Classificação: Prevendo a Degenerescência Espectral
Hoje o alvo deixa de ser um número contínuo e passa a ser uma categoria: uma camada domínio-ano do acervo é "espectralmente degenerada" (topologia quase-estrela, |λmin|/λ1 > 0,9) ou não? É a mesma métrica que o artigo original usa na Seção 5 — e o desbalanceamento de classes aqui é real, não fabricado para o exercício.

## **recarregar o acervo — padrão validado**

In [1]:
import requests, zipfile, io, os
import pandas as pd
import numpy as np

url = (
    "https://zenodo.org/records/21211053/files/"
    "openfcs_v1.0.0.zip?download=1"
)
resp = requests.get(url)
resp.raise_for_status()

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    z.extractall("openfcs")

endereco = "openfcs/openfcs-1.0.0/data/derived/"
edges = pd.read_csv(endereco + "trade_edges.csv")
edges7 = edges[edges["resolution"] == "cer7"].copy()
spectral = pd.read_csv(endereco + "spectral_results.csv")

print(f"edges7:   {len(edges7):,} linhas")
print(f"spectral: {len(spectral):,} linhas")

edges7:   1,026,400 linhas
spectral: 322 linhas


### 10.1 Construindo o alvo binário: degenerescência espectral
A regra do artigo (Capítulo 1 e Seção 5): uma camada é degenerada quando |λmin|/λ1 > 0,9. Vamos calcular isso nós mesmos, sobre a resolução cer7, e conferir se bate com os "14 de 161" que o artigo reporta.

In [2]:
spectral7 = spectral[spectral["resolution"] == "cer7"].copy()

spectral7["razao_degenerescencia"] = (
    spectral7["lambda_min"].abs() / spectral7["lambda1"]
)
spectral7["degenerado"] = (
    spectral7["razao_degenerescencia"] > 0.9
).astype(int)

print(f"Total de camadas (cer7): {len(spectral7)}")
print(spectral7["degenerado"].value_counts())
print("\nReferencia do artigo: 14 de 161 camadas degeneradas")

Total de camadas (cer7): 161
degenerado
0    147
1     14
Name: count, dtype: int64

Referencia do artigo: 14 de 161 camadas degeneradas


### 10.2 O problema do desbalanceamento
Se a proporção acima ficou perto de 14/161 (≈ 8,7%), estamos diante de um desbalanceamento real: a classe minoritária ("degenerado") é rara. Isso muda tudo — inclusive qual métrica de avaliação faz sentido usar.

In [3]:
exportador_dominio_ano = (
    edges7.groupby(["fcs_domain", "year", "economy"])["value_usd_millions"]
    .sum()
    .reset_index(name="total_economia")
)
total_camada = exportador_dominio_ano.groupby(["fcs_domain", "year"])[
    "total_economia"
].transform("sum")
exportador_dominio_ano["participacao"] = (
    exportador_dominio_ano["total_economia"] / total_camada
)
hhi_por_dominio_ano = (
    exportador_dominio_ano.groupby(["fcs_domain", "year"])["participacao"]
    .apply(lambda s: (s ** 2).sum())
    .reset_index(name="HHI")
)
hhi_por_dominio_ano.head()

,fcs_domain,year,HHI
0,A. Cultural and natural heritage,2002,0.307426
1,A. Cultural and natural heritage,2003,0.404968
2,A. Cultural and natural heritage,2004,0.259860
3,A. Cultural and natural heritage,2005,0.259741
4,A. Cultural and natural heritage,2006,0.219693


### 10.3 Preparando as features: cuidado com o vazamento
`spectral_entropy` e `lcc_fraction` vêm do mesmo cálculo espectral que originou o alvo — usá-las é um risco de vazamento (o modelo pode "colar" em vez de aprender um padrão real). Vamos construir dois conjuntos de features: um limpo, outro deliberadamente vazado, para comparar depois.

In [4]:
base = spectral7.merge(
    hhi_por_dominio_ano, on=["fcs_domain", "year"], how="inner"
)
print(f"Linhas apos merge com HHI: {len(base)}")

features_limpas = ["fcs_domain", "year", "n_economies", "n_edges", "HHI"]
features_vazadas = features_limpas + ["spectral_entropy", "lcc_fraction"]

X_limpo = base[features_limpas]
X_vazado = base[features_vazadas]
y = base["degenerado"]

Linhas apos merge com HHI: 161


### 10.4 Divisão treino/teste estratificada
Com apenas ~14 casos positivos em 161 linhas, um split aleatório comum arrisca deixar poucos (ou nenhum) exemplo da classe minoritária no teste. `stratify=y` garante a mesma proporção em treino e teste.

In [5]:
from sklearn.model_selection import train_test_split

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X_limpo, y, test_size=0.25, stratify=y, random_state=42
)

print(f"Treino: {len(X_treino)} linhas, {y_treino.sum()} degenerados")
print(f"Teste:  {len(X_teste)} linhas, {y_teste.sum()} degenerados")

Treino: 120 linhas, 10 degenerados
Teste:  41 linhas, 4 degenerados


### 10.5 Pipeline e regressão logística
`class_weight="balanced"` ajusta o peso de cada classe automaticamente, compensando o desbalanceamento sem precisar duplicar ou remover linhas.

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

pre_processador = ColumnTransformer(transformers=[
    (
        "categorica",
        OneHotEncoder(handle_unknown="ignore"),
        ["fcs_domain"],
    ),
    (
        "numerica",
        StandardScaler(),
        ["year", "n_economies", "n_edges", "HHI"],
    ),
])

modelo_logistico = Pipeline(steps=[
    ("preprocessamento", pre_processador),
    ("modelo", LogisticRegression(class_weight="balanced")),
])

modelo_logistico.fit(X_treino, y_treino)
y_pred_log = modelo_logistico.predict(X_teste)

### 10.6 Matriz de confusão e métricas para dados desbalanceados
Acurácia sozinha engana em dados desbalanceados. Precision, recall e F1 contam a história completa.

In [7]:
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score
)

print("Matriz de confusao:")
print(confusion_matrix(y_teste, y_pred_log))
print(f"\nAcuracia: {accuracy_score(y_teste, y_pred_log):.3f}")
print("\n", classification_report(y_teste, y_pred_log, zero_division=0))

Matriz de confusao:
[[30  7]
 [ 2  2]]

Acuracia: 0.780

               precision    recall  f1-score   support

           0       0.94      0.81      0.87        37
           1       0.22      0.50      0.31         4

    accuracy                           0.78        41
   macro avg       0.58      0.66      0.59        41
weighted avg       0.87      0.78      0.81        41



### 10.7 A armadilha da acurácia: um "modelo" que nunca erra por preguiça

In [8]:
from sklearn.dummy import DummyClassifier

modelo_bobo = DummyClassifier(strategy="most_frequent")
modelo_bobo.fit(X_treino, y_treino)
y_pred_bobo = modelo_bobo.predict(X_teste)

print(f"Acuracia do modelo bobo (sempre 'nao degenerado'):")
print(f"{accuracy_score(y_teste, y_pred_bobo):.3f}")
print(f"\nMas o recall da classe degenerada eh sempre 0 -")
print(f"o modelo bobo nunca acerta o que realmente importa.")

Acuracia do modelo bobo (sempre 'nao degenerado'):
0.902

Mas o recall da classe degenerada eh sempre 0 -
o modelo bobo nunca acerta o que realmente importa.


### 10.8 KNN

In [9]:
from sklearn.neighbors import KNeighborsClassifier

modelo_knn = Pipeline(steps=[
    ("preprocessamento", pre_processador),
    ("modelo", KNeighborsClassifier(n_neighbors=5)),
])
modelo_knn.fit(X_treino, y_treino)
y_pred_knn = modelo_knn.predict(X_teste)

print(classification_report(y_teste, y_pred_knn, zero_division=0))

              precision    recall  f1-score   support

           0       0.90      1.00      0.95        37
           1       0.00      0.00      0.00         4

    accuracy                           0.90        41
   macro avg       0.45      0.50      0.47        41
weighted avg       0.81      0.90      0.86        41



### 10.9 Árvore de decisão

In [10]:
from sklearn.tree import DecisionTreeClassifier

modelo_arvore = Pipeline(steps=[
    ("preprocessamento", pre_processador),
    (
        "modelo",
        DecisionTreeClassifier(
            max_depth=4, class_weight="balanced", random_state=42
        ),
    ),
])
modelo_arvore.fit(X_treino, y_treino)
y_pred_arvore = modelo_arvore.predict(X_teste)

print(classification_report(y_teste, y_pred_arvore, zero_division=0))

              precision    recall  f1-score   support

           0       0.97      0.89      0.93        37
           1       0.43      0.75      0.55         4

    accuracy                           0.88        41
   macro avg       0.70      0.82      0.74        41
weighted avg       0.92      0.88      0.89        41



### 10.10 O efeito do vazamento, medido de verdade
Repetindo a regressão logística com as features "vazadas" (`spectral_entropy`, `lcc_fraction`). Se o desempenho disparar, é sinal de alerta, não de vitória.

In [11]:
X_treino_vz, X_teste_vz = X_treino.copy(), X_teste.copy()
X_treino_vz = X_vazado.loc[X_treino.index]
X_teste_vz = X_vazado.loc[X_teste.index]

pre_processador_vz = ColumnTransformer(transformers=[
    (
        "categorica",
        OneHotEncoder(handle_unknown="ignore"),
        ["fcs_domain"],
    ),
    (
        "numerica",
        StandardScaler(),
        ["year", "n_economies", "n_edges", "HHI",
         "spectral_entropy", "lcc_fraction"],
    ),
])

modelo_vazado = Pipeline(steps=[
    ("preprocessamento", pre_processador_vz),
    ("modelo", LogisticRegression(class_weight="balanced")),
])
modelo_vazado.fit(X_treino_vz, y_treino)
y_pred_vazado = modelo_vazado.predict(X_teste_vz)

print(classification_report(y_teste, y_pred_vazado, zero_division=0))

              precision    recall  f1-score   support

           0       0.97      0.81      0.88        37
           1       0.30      0.75      0.43         4

    accuracy                           0.80        41
   macro avg       0.63      0.78      0.66        41
weighted avg       0.90      0.80      0.84        41



### 10.11 Validação mais robusta: k-fold estratificado
Um único split de teste, com poucos positivos, dá uma estimativa instável. `StratifiedKFold` avalia em várias partições e reporta a média.

In [12]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

kfold_estrat = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = cross_val_score(
    modelo_logistico, X_limpo, y, cv=kfold_estrat, scoring="f1"
)
print(f"F1 por fold: {np.round(f1_scores, 3)}")
print(f"F1 medio:    {f1_scores.mean():.3f} (+/- {f1_scores.std():.3f})")

F1 por fold: [0.333 0.286 0.5   0.364 0.75 ]
F1 medio:    0.447 (+/- 0.168)


### 10.12 Comparativo final

In [13]:
from sklearn.metrics import f1_score, precision_score, recall_score

def metricas_clf(y_real, y_prev, nome):
    return {
        "modelo": nome,
        "acuracia": accuracy_score(y_real, y_prev),
        "precision": precision_score(y_real, y_prev, zero_division=0),
        "recall": recall_score(y_real, y_prev, zero_division=0),
        "f1": f1_score(y_real, y_prev, zero_division=0),
    }

resultados = pd.DataFrame([
    metricas_clf(y_teste, y_pred_bobo, "Bobo (sempre 0)"),
    metricas_clf(y_teste, y_pred_log, "Regressao logistica"),
    metricas_clf(y_teste, y_pred_knn, "KNN (k=5)"),
    metricas_clf(y_teste, y_pred_arvore, "Arvore de decisao"),
    metricas_clf(y_teste, y_pred_vazado, "Logistica (com vazamento)"),
])
resultados.sort_values("f1", ascending=False)

,modelo,acuracia,precision,recall,f1
3,Arvore de decisao,0.878049,0.428571,0.75,0.545455
4,Logistica (com vazamento),0.804878,0.300000,0.75,0.428571
1,Regressao logistica,0.780488,0.222222,0.50,0.307692
0,Bobo (sempre 0),0.902439,0.000000,0.00,0.000000
2,KNN (k=5),0.902439,0.000000,0.00,0.000000


## 🧪 Exercícios Práticos — Aula 10

> **Como usar:** resolva cada exercício em uma célula de código abaixo do enunciado. Depois, leve o resultado para uma IA usando o *prompt sugerido*.

---

### Exercício 1 — Em quais domínios a degenerescência se concentra?
**📝 Tarefa:** Filtre `spectral7` para as linhas com `degenerado == 1` e conte quantas caem em cada `fcs_domain`. Isso bate com o que o artigo relata (concentração em patrimônio e arquitetura)?

**🤖 Pergunte à IA:**
> "As camadas degeneradas do meu dataset se concentram nestes domínios: [cole a contagem]. Do ponto de vista de teoria de redes, por que domínios com poucos exportadores tenderiam a gerar topologias quase-estrela?"

---

### Exercício 2 — Ajustando o `k` do KNN
**📝 Tarefa:** Repita a Célula 18 com `n_neighbors=3` e `n_neighbors=15`. O F1 da classe degenerada melhora, piora, ou não muda de forma clara?

**🤖 Pergunte à IA:**
> "Testei KNN com k=3, k=5 e k=15 em um problema de classificação desbalanceado (91%/9%). Os F1 da classe minoritária foram [X, Y, Z]. Por que k muito alto tende a prejudicar a classe minoritária nesse tipo de problema?"

---

### Exercício 3 — Visualizando a árvore de decisão
**📝 Tarefa:** Use `sklearn.tree.plot_tree` para desenhar a árvore da Célula 20. Qual é a primeira pergunta (nó raiz) que a árvore faz para decidir se uma camada é degenerada?

**🤖 Pergunte à IA:**
> "A primeira divisão da minha árvore de decisão usa a variável [X] com o limiar [Y]. Isso faz sentido do ponto de vista de como uma topologia de rede se torna degenerada?"

---

### Exercício 4 — O tamanho do vazamento
**📝 Tarefa:** Compare, lado a lado, o F1 da regressão logística limpa (Célula 14) com o da versão vazada (Célula 22). Qual `feature` vazada parece explicar a maior parte do ganho artificial?

**🤖 Pergunte à IA:**
> "Meu modelo limpo teve F1 de [X]; a versão com features potencialmente vazadas teve F1 de [Y]. Que tipo de auditoria eu deveria fazer antes de confiar em um ganho de desempenho tão grande?"

---

### Exercício 5 — Ajustando o limiar de decisão
**📝 Tarefa:** Em vez de usar `.predict()` (limiar padrão de 0,5), use `.predict_proba()` da regressão logística e teste um limiar mais baixo (ex.: 0,3) para classificar como degenerado. O recall da classe minoritária melhora? O que se perde em troca?

**🤖 Pergunte à IA:**
> "Ao reduzir o limiar de decisão de 0,5 para 0,3 em um classificador binário, meu recall foi de [X] para [Y] e minha precision foi de [W] para [Z]. Em que situação de negócio esse tipo de troca vale a pena?"